# Tokenization: BPE From Scratch, Real Tokenizers, and CLM vs MLM

---

Creator: Chung-En Johnny Yu

Content updated: 2026-10-29

Live companion for **M15 — Tokenization + Attention** (tokenization half). The course notebook `01_tokenization.ipynb` (Raschka Ch. 2) builds a simple tokenizer, uses GPT-2's BPE via `tiktoken`, samples input–target pairs and creates embeddings. This companion opens the black box of BPE, shows what real tokenizers do to code, numbers and other languages, and connects token ids to the two classic training objectives.

**Sources** — every section is tagged:

| Tag | Source | License |
|---|---|---|
| 📗 **[BPE bonus]** | Sebastian Raschka, LLMs-from-scratch [`ch02/05_bpe-from-scratch/bpe-from-scratch-simple.ipynb`](https://github.com/rasbt/LLMs-from-scratch/blob/main/ch02/05_bpe-from-scratch/bpe-from-scratch-simple.ipynb) | Apache-2.0 (code/text) |
| 📗 **[Embedding bonus]** | Sebastian Raschka, LLMs-from-scratch [`ch02/03_bonus_embedding-vs-matmul`](https://github.com/rasbt/LLMs-from-scratch/tree/main/ch02/03_bonus_embedding-vs-matmul) | Apache-2.0 (code/text) |
| 📗 **[LLMs-from-scratch Ch2]** | Sebastian Raschka, *Build a Large Language Model (From Scratch)* (Manning, 2024), Ch. 2 figures | book figures are not covered by the repo license; copied into `01-tokenization_clm_mlm/img/` with attribution (see `SOURCE.md`) |
| 🧪 **[Added]** | Written for this companion | — |

Runs on CPU in Colab in under a minute. The first code cell installs `tiktoken` if needed.

| Part | Content |
|---|---|
| 1 | Why tokenize, and the vocabulary-size trade-off |
| 2 | Bits and bytes: where byte-level BPE starts |
| 3 | The BPE algorithm, step by step |
| 4 | A simple BPE implementation: train, encode, decode |
| 5 | Real tokenizers: GPT-2 vs GPT-4 vocabularies and their quirks |
| 6 | From token ids to vectors: embedding = one-hot × matrix |
| 7 | What the ids are trained to predict: causal LM vs masked LM |
| 8 | Interview-style check questions |

---

In [1]:
import os, sys, math, subprocess, urllib.request
try:
    import tiktoken
except ImportError:                                  # Colab: install once
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "tiktoken"])
    import tiktoken
import torch
import torch.nn.functional as F

print("torch", torch.__version__, "| tiktoken", tiktoken.__version__)

torch 2.13.0 | tiktoken 0.13.0


## 1. Why tokenize, and the trade-off

📗 *[BPE bonus] figure; 🧪 [Added] text*

A model only sees integers. The **tokenizer** maps text → a sequence of ids from a fixed vocabulary of size $V$, and back again after generation:

<img src="https://github.com/chungenyu6/ai-learning-notebooks/blob/main/02-LLM/live/01-tokenization_clm_mlm/img/bpe-overview.webp?raw=true" width="550px">

*Figure: where the tokenizer sits in an LLM pipeline. Source: Raschka, LLMs-from-scratch, BPE-from-scratch bonus.*

| Unit | Vocabulary $V$ | Sequence length $T$ | Problem |
|---|---|---|---|
| Characters / bytes | tiny (256 bytes) | very long | attention costs $O(T^2)$; each step carries little meaning |
| Words | huge, open-ended | short | out-of-vocabulary words (`<unk>`), no sharing between *run / running / runner* |
| **Subwords (BPE, WordPiece, Unigram)** | 30k–200k | moderate | the standard compromise: frequent words are one token, rare words split into pieces |

$V$ also costs parameters: the input embedding and the output layer are each $V\times d$. For GPT-2 ($V=50{,}257$, $d=768$) that is ~38.6M parameters per matrix — about a third of the 124M model (GPT-2 ties the two, so it pays once).

## 2. Bits and bytes

📗 *[BPE bonus] text and code*

- Before getting to the BPE algorithm, let's introduce the notion of bytes
- Consider converting text into a byte array (BPE stands for "byte" pair encoding after all):

In [2]:
# Source: Raschka, LLMs-from-scratch, bpe-from-scratch-simple.ipynb (Apache-2.0)
text = "This is some text"
byte_ary = bytearray(text, "utf-8")
print(byte_ary)

ids = list(byte_ary)
print(ids)

print("Number of characters:", len(text))
print("Number of token IDs:", len(ids))

bytearray(b'This is some text')
[84, 104, 105, 115, 32, 105, 115, 32, 115, 111, 109, 101, 32, 116, 101, 120, 116]
Number of characters: 17
Number of token IDs: 17


- This would be a valid way to convert text into token IDs, but it creates one ID per character — a lot of IDs for a short text
- BPE tokenizers instead have a vocabulary with token IDs for whole words or subwords. The GPT-2 tokenizer turns the same text into only 4 tokens, `1212, 318, 617, 2420` — you can check this in the interactive [tiktokenizer app](https://tiktokenizer.vercel.app/?model=gpt2):

<img src="https://github.com/chungenyu6/ai-learning-notebooks/blob/main/02-LLM/live/01-tokenization_clm_mlm/img/tiktokenizer.webp?raw=true" width="600px">

*Figure: GPT-2 tokenization of "This is some text". Source: Raschka, LLMs-from-scratch, BPE-from-scratch bonus.*

- Since a byte has 8 bits there are $2^8=256$ possible values (0–255). A byte-level BPE tokenizer uses these 256 values as its first 256 tokens, so **any** input text can be encoded — there is never an out-of-vocabulary token.
- In GPT-2, entries 256 and 257 are already two-character tokens (a space plus a letter):

In [3]:
# Source: adapted from Raschka, bpe-from-scratch-simple.ipynb (Apache-2.0)
gpt2_tokenizer = tiktoken.get_encoding("gpt2")
print(gpt2_tokenizer.encode("This is some text"))
for i in [0, 1, 2, 255, 256, 257, 298]:
    print(f"{i}: {gpt2_tokenizer.decode([i])!r}")

[1212, 318, 617, 2420]
0: '!'
1: '"'
2: '#'
255: '�'
256: ' t'
257: ' a'
298: 'ent'


## 3. The BPE algorithm

📗 *[BPE bonus] text*

The goal of BPE is to build a vocabulary of commonly occurring subwords like `298: ent` (found in *entangle, entertain, enter, entrance, entity, ...*) or even complete words like `318: is`, `617: some`, `1212: This`, `2420: text`. The algorithm was originally described in 1994 as a data-compression method ([Gage, *A New Algorithm for Data Compression*](https://github.com/tpn/pdfs/blob/master/A%20New%20Algorithm%20for%20Data%20Compression%20(1994).pdf)).

**1. Identify frequent pairs** — in each iteration, scan the text to find the most commonly occurring pair of bytes (or characters).

**2. Replace and record** — replace that pair with a new placeholder ID (one not already in use; starting from 0…255 the first is 256), and record the mapping in a lookup table. The size of the lookup table is a hyperparameter, the **vocabulary size** (50,257 for GPT-2).

**3. Repeat until no gains** — keep merging the most frequent pairs; stop when no further compression is possible or the target vocabulary size is reached.

**Decompression (decoding)** — restore the original text by substituting each ID with its pair, using the lookup table.

**Worked example** on the training text `the cat in the hat`:

| Iteration | Most frequent pair | New id | Text after merge |
|---|---|---|---|
| 1 | `t` `h` (twice) | 256 = "th" | `<256>e cat in <256>e hat` |
| 2 | `<256>` `e` | 257 = "<256>e" | `<257> cat in <257> hat` |
| 3 | `<257>` `␣` | 258 = "<257> " | `<258>cat in <258>hat` |

Decoding reverses the substitutions in reverse order: `<258>` → `<257> ` → `<256>e ` → `the `.

## 4. A simple BPE implementation

📗 *[BPE bonus] text and code*

- Below is an implementation of this algorithm as a Python class that mimics the `tiktoken` Python user interface
- The encoding part above describes the training step via `train()`; the `encode()` method works similarly (although it looks a bit more complicated because of the special token handling):
  1. Split the input text into individual bytes
  2. Repeatedly find & replace (merge) adjacent tokens (pairs) when they match any pair in the learned BPE merges (from highest to lowest "rank," i.e., in the order they were learned)
  3. Continue merging until no more merges can be applied
  4. The final list of token IDs is the encoded output

This is a deliberately naive, readable implementation for education; Raschka's `bpe-from-scratch.ipynb` has a faster version that matches `tiktoken` exactly. You do not need to read every line — the walkthrough below shows how to use it.

In [4]:
# Source: Raschka, LLMs-from-scratch, bpe-from-scratch-simple.ipynb (Apache-2.0)
from collections import Counter, deque
from functools import lru_cache


class BPETokenizerSimple:
    def __init__(self):
        # Maps token_id to token_str (e.g., {11246: "some"})
        self.vocab = {}
        # Maps token_str to token_id (e.g., {"some": 11246})
        self.inverse_vocab = {}
        # Dictionary of BPE merges: {(token_id1, token_id2): merged_token_id}
        self.bpe_merges = {}

    def train(self, text, vocab_size, allowed_special={"<|endoftext|>"}):
        """
        Train the BPE tokenizer from scratch.

        Args:
            text (str): The training text.
            vocab_size (int): The desired vocabulary size.
            allowed_special (set): A set of special tokens to include.
        """

        # Preprocess: Replace spaces with 'Ġ'
        # Note that Ġ is a particularity of the GPT-2 BPE implementation
        # E.g., "Hello world" might be tokenized as ["Hello", "Ġworld"]
        # (GPT-4 BPE would tokenize it as ["Hello", " world"])
        processed_text = []
        for i, char in enumerate(text):
            if char == " " and i != 0:
                processed_text.append("Ġ")
            if char != " ":
                processed_text.append(char)
        processed_text = "".join(processed_text)

        # Initialize vocab with unique characters, including 'Ġ' if present
        # Start with the first 256 ASCII characters
        unique_chars = [chr(i) for i in range(256)]

        # Extend unique_chars with characters from processed_text that are not already included
        unique_chars.extend(char for char in sorted(set(processed_text)) if char not in unique_chars)

        # Optionally, ensure 'Ġ' is included if it is relevant to your text processing
        if 'Ġ' not in unique_chars:
            unique_chars.append('Ġ')

        # Now create the vocab and inverse vocab dictionaries
        self.vocab = {i: char for i, char in enumerate(unique_chars)}
        self.inverse_vocab = {char: i for i, char in self.vocab.items()}

        # Add allowed special tokens
        if allowed_special:
            for token in allowed_special:
                if token not in self.inverse_vocab:
                    new_id = len(self.vocab)
                    self.vocab[new_id] = token
                    self.inverse_vocab[token] = new_id

        # Tokenize the processed_text into token IDs
        token_ids = [self.inverse_vocab[char] for char in processed_text]

        # BPE steps 1-3: Repeatedly find and replace frequent pairs
        for new_id in range(len(self.vocab), vocab_size):
            if len(token_ids) < 2:
                break
            pair_id = self.find_freq_pair(token_ids, mode="most")
            if pair_id is None:  # No more pairs to merge. Stopping training.
                break
            
            updated = self.replace_pair(token_ids, pair_id, new_id)
            if updated == token_ids:
                break

            token_ids = updated
            self.bpe_merges[pair_id] = new_id

        # Build the vocabulary with merged tokens
        for (p0, p1), new_id in self.bpe_merges.items():
            merged_token = self.vocab[p0] + self.vocab[p1]
            self.vocab[new_id] = merged_token
            self.inverse_vocab[merged_token] = new_id

    def encode(self, text):
        """
        Encode the input text into a list of token IDs.

        Args:
            text (str): The text to encode.

        Returns:
            List[int]: The list of token IDs.
        """
        tokens = []
        # Split text into tokens, keeping newlines intact
        words = text.replace("\n", " \n ").split()  # Ensure '\n' is treated as a separate token

        for i, word in enumerate(words):
            if i > 0 and not word.startswith("\n"):
                tokens.append("Ġ" + word)  # Add 'Ġ' to words that follow a space or newline
            else:
                tokens.append(word)  # Handle first word or standalone '\n'

        token_ids = []
        for token in tokens:
            if token in self.inverse_vocab:
                # token is contained in the vocabulary as is
                token_id = self.inverse_vocab[token]
                token_ids.append(token_id)
            else:
                # Attempt to handle subword tokenization via BPE
                sub_token_ids = self.tokenize_with_bpe(token)
                token_ids.extend(sub_token_ids)

        return token_ids

    def tokenize_with_bpe(self, token):
        """
        Tokenize a single token using BPE merges.

        Args:
            token (str): The token to tokenize.

        Returns:
            List[int]: The list of token IDs after applying BPE.
        """
        # Tokenize the token into individual characters (as initial token IDs)
        token_ids = [self.inverse_vocab.get(char, None) for char in token]
        if None in token_ids:
            missing_chars = [char for char, tid in zip(token, token_ids) if tid is None]
            raise ValueError(f"Characters not found in vocab: {missing_chars}")

        can_merge = True
        while can_merge and len(token_ids) > 1:
            can_merge = False
            new_tokens = []
            i = 0
            while i < len(token_ids) - 1:
                pair = (token_ids[i], token_ids[i + 1])
                if pair in self.bpe_merges:
                    merged_token_id = self.bpe_merges[pair]
                    new_tokens.append(merged_token_id)
                    # Uncomment for educational purposes:
                    # print(f"Merged pair {pair} -> {merged_token_id} ('{self.vocab[merged_token_id]}')")
                    i += 2  # Skip the next token as it's merged
                    can_merge = True
                else:
                    new_tokens.append(token_ids[i])
                    i += 1
            if i < len(token_ids):
                new_tokens.append(token_ids[i])
            token_ids = new_tokens

        return token_ids

    def decode(self, token_ids):
        """
        Decode a list of token IDs back into a string.

        Args:
            token_ids (List[int]): The list of token IDs to decode.

        Returns:
            str: The decoded string.
        """
        decoded_string = ""
        for token_id in token_ids:
            if token_id not in self.vocab:
                raise ValueError(f"Token ID {token_id} not found in vocab.")
            token = self.vocab[token_id]
            if token.startswith("Ġ"):
                # Replace 'Ġ' with a space
                decoded_string += " " + token[1:]
            else:
                decoded_string += token
        return decoded_string

    @lru_cache(maxsize=None)
    def get_special_token_id(self, token):
        return self.inverse_vocab.get(token, None)

    @staticmethod
    def find_freq_pair(token_ids, mode="most"):
        if(len(token_ids) < 2):
            return None
        pairs = Counter(zip(token_ids, token_ids[1:]))
        if not pairs:
            return None

        if mode == "most":
            return max(pairs.items(), key=lambda x: x[1])[0]
        elif mode == "least":
            return min(pairs.items(), key=lambda x: x[1])[0]
        else:
            raise ValueError("Invalid mode. Choose 'most' or 'least'.")

    @staticmethod
    def replace_pair(token_ids, pair_id, new_id):
        dq = deque(token_ids)
        replaced = []

        while dq:
            current = dq.popleft()
            if dq and (current, dq[0]) == pair_id:
                replaced.append(new_id)
                # Remove the 2nd token of the pair, 1st was already removed
                dq.popleft()
            else:
                replaced.append(current)

        return replaced

### Training, encoding and decoding

The training text is *The Verdict*, a short story by Edith Wharton (public domain) used throughout Raschka's Chapter 2. We train a vocabulary of 1,000: the first 256 entries are the byte values, so ~745 entries are actually *learned*. For comparison, GPT-2 has 50,257 tokens, GPT-4 (`cl100k_base`) about 100k and GPT-4o (`o200k_base`) about 200k.

In [5]:
# 🧪 [Added] download helper: local copy -> this repo -> original book repo
fname = "the-verdict.txt"
local = os.path.join("01-tokenization_clm_mlm", "data", fname)
if os.path.exists(local):
    fname = local
elif not os.path.exists(fname):
    for url in ["https://raw.githubusercontent.com/chungenyu6/ai-learning-notebooks/main/02-LLM/live/01-tokenization_clm_mlm/data/the-verdict.txt",
                "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/main/ch02/01_main-chapter-code/the-verdict.txt"]:
        try:
            urllib.request.urlretrieve(url, fname); break
        except Exception as e:
            print("could not download", url, e)

with open(fname, "r", encoding="utf-8") as f:
    text = f.read()
print("characters:", len(text))

characters: 20479


In [6]:
# Source: Raschka, LLMs-from-scratch, bpe-from-scratch-simple.ipynb (Apache-2.0)
tokenizer = BPETokenizerSimple()
tokenizer.train(text, vocab_size=1000, allowed_special={"<|endoftext|>"})

print(len(tokenizer.vocab))
print(len(tokenizer.bpe_merges))

1000
742


In [7]:
# Source: Raschka, LLMs-from-scratch, bpe-from-scratch-simple.ipynb (Apache-2.0)
input_text = "Jack embraced beauty through art and life."
token_ids = tokenizer.encode(input_text)
print(token_ids)
print("Number of characters:", len(input_text))
print("Number of token IDs:", len(token_ids))

for token_id in token_ids:
    print(f"{token_id} -> {tokenizer.decode([token_id])}")

[424, 256, 654, 531, 302, 311, 256, 296, 97, 465, 121, 595, 841, 116, 287, 466, 256, 326, 972, 46]
Number of characters: 42
Number of token IDs: 20
424 -> Jack
256 ->  
654 -> em
531 -> br
302 -> ac
311 -> ed
256 ->  
296 -> be
97 -> a
465 -> ut
121 -> y
595 ->  through
841 ->  ar
116 -> t
287 ->  a
466 -> nd
256 ->  
326 -> li
972 -> fe
46 -> .


- A 42-character sentence becomes about 20 token IDs — roughly half the length of a byte encoding.
- Most tokens are 2-character subwords because the training text is short and the vocabulary small.
- `decode(encode(x))` reproduces arbitrary input:

In [8]:
# Source: Raschka, bpe-from-scratch-simple.ipynb (Apache-2.0); 🧪 assert added
print(tokenizer.decode(tokenizer.encode("This is some text.")))
for s in ["This is some text.", "Unseen words: zebra, quokka!", "naïve café"]:
    assert tokenizer.decode(tokenizer.encode(s)) == s
print("round trip on unseen words ✓")

# 🧪 [Added] a limitation of the *simple* implementation
try:
    tokenizer.encode("注意力")
except ValueError as e:
    print("simple BPE fails on CJK:", e)
print("tiktoken (true byte-level) handles it:", gpt2_tokenizer.encode("注意力"))

This is some text.
round trip on unseen words ✓
simple BPE fails on CJK: Characters not found in vocab: ['注', '意', '力']
tiktoken (true byte-level) handles it: [37345, 101, 35707, 237, 27950, 249]


🧪 **Why did Chinese fail?** This simple class initializes its base vocabulary with the first 256 *Unicode characters* (`chr(0)`…`chr(255)`), which covers ASCII and accented Latin letters but not, e.g., 注. A real **byte-level** BPE (GPT-2, `tiktoken`, and Raschka's full `bpe-from-scratch.ipynb`) first converts text to UTF-8 **bytes**, so every possible string is made of the 256 base tokens and nothing is ever out of vocabulary.

🧪 **Look at what was learned.** The first merges are the most frequent pairs in the training text. `Ġ` is how GPT-2-style tokenizers display a **space**: pairs like `eĠ` / `sĠ` capture word endings and `Ġt` captures word starts, which is why so many GPT tokens begin with a space.

In [9]:
# 🧪 [Added]
first = list(tokenizer.bpe_merges.items())[:12]
for (a, b), new_id in first:
    print(f"merge {new_id}: {tokenizer.vocab[a]!r} + {tokenizer.vocab[b]!r} -> {tokenizer.vocab[new_id]!r}")

merge 258: 'e' + 'Ġ' -> 'eĠ'
merge 259: 'Ġ' + 't' -> 'Ġt'
merge 260: 'd' + 'Ġ' -> 'dĠ'
merge 261: 't' + 'Ġ' -> 'tĠ'
merge 262: 'i' + 'n' -> 'in'
merge 263: 's' + 'Ġ' -> 'sĠ'
merge 264: 'h' + 'eĠ' -> 'heĠ'
merge 265: 'h' + 'a' -> 'ha'
merge 266: ',' + 'Ġ' -> ',Ġ'
merge 267: 'o' + 'u' -> 'ou'
merge 268: 'e' + 'r' -> 'er'
merge 269: 'a' + 'n' -> 'an'


Unknown words are not a problem — they are split into known subwords or, at worst, single bytes:

<img src="https://github.com/chungenyu6/ai-learning-notebooks/blob/main/02-LLM/live/01-tokenization_clm_mlm/img/ch02_11.webp?raw=true" width="550px">

*Figure: BPE breaks an unknown word into subword units. Source: Raschka, LLMs-from-scratch Ch2 (original image `ch02_compressed/11.webp`).*

In [10]:
# 🧪 [Added]
for w in ["Akwirw ier", "unbelievably", "LLMs"]:
    ids_ = gpt2_tokenizer.encode(w)
    print(f"{w!r:16} -> {[gpt2_tokenizer.decode([i]) for i in ids_]}")

'Akwirw ier'     -> ['Ak', 'w', 'ir', 'w', ' ', 'ier']
'unbelievably'   -> ['un', 'bel', 'iev', 'ably']
'LLMs'           -> ['LL', 'Ms']


## 5. Real tokenizers and their quirks

🧪 *[Added]*

Compare GPT-2's vocabulary (50,257) with the GPT-4-era `cl100k_base` (~100k):

In [11]:
# 🧪 [Added]
cl100k = tiktoken.get_encoding("cl100k_base")

def show(enc, text):
    ids_ = enc.encode(text, allowed_special="all")
    print(f"{enc.name:12s} {len(ids_):3d} tokens: {[enc.decode([i]) for i in ids_]}")

for sample in ["Tokenizers aren't magic: unbelievably, they're just compression.",
               "    def forward(self, x):",
               "1234567 + 7654321 = 8888888",
               "注意力機制 attention",
               "<|endoftext|>"]:
    print(repr(sample))
    show(gpt2_tokenizer, sample); show(cl100k, sample); print()

"Tokenizers aren't magic: unbelievably, they're just compression."
gpt2          13 tokens: ['Token', 'izers', ' aren', "'t", ' magic', ':', ' unbelievably', ',', ' they', "'re", ' just', ' compression', '.']
cl100k_base   14 tokens: ['Token', 'izers', ' aren', "'t", ' magic', ':', ' unbelie', 'vably', ',', ' they', "'re", ' just', ' compression', '.']

'    def forward(self, x):'
gpt2          10 tokens: [' ', ' ', ' ', ' def', ' forward', '(', 'self', ',', ' x', '):']
cl100k_base    7 tokens: ['   ', ' def', ' forward', '(self', ',', ' x', '):']

'1234567 + 7654321 = 8888888'
gpt2          13 tokens: ['123', '45', '67', ' +', ' 7', '65', '43', '21', ' =', ' 8', '88', '88', '88']
cl100k_base   13 tokens: ['123', '456', '7', ' +', ' ', '765', '432', '1', ' =', ' ', '888', '888', '8']

'注意力機制 attention'
gpt2          10 tokens: ['�', '�', '�', '�', '�', '�', '機', '�', '�', ' attention']
cl100k_base    7 tokens: ['注意', '力', '�', '�', '�', '制', ' attention']

'<|endoftext|>'
gpt2         

Each of these is a classic interview follow-up:

- **Leading spaces are part of the token** (`' magic'`): `"Hello"` and `" Hello"` are different ids.
- **Code whitespace**: GPT-2 spends one token per space; the larger vocabulary merges runs of spaces — a larger vocabulary is cheaper for code.
- **Numbers** are chunked irregularly — one reason LLMs are weak at digit-level arithmetic.
- **Non-English text** costs more tokens per character under an English-heavy vocabulary → higher cost and shorter effective context. The `�` pieces are tokens that hold only *part* of a multi-byte UTF-8 character; only the full sequence decodes to valid text.
- **Special tokens** like `<|endoftext|>` are single ids marking document boundaries; chat models add role markers (M18).

In [12]:
# 🧪 [Added]
for s in ["注意力機制 attention", "🙂 emoji"]:
    assert gpt2_tokenizer.decode(gpt2_tokenizer.encode(s)) == s
print("byte-level BPE round-trips arbitrary text ✓")
print("one emoji, two byte-fragment tokens:", [gpt2_tokenizer.decode_single_token_bytes(i) for i in gpt2_tokenizer.encode("🙂")])

byte-level BPE round-trips arbitrary text ✓
one emoji, two byte-fragment tokens: [b'\xf0\x9f', b'\x99\x82']


## 6. From token ids to vectors

📗 *[Embedding bonus] text, code and figures; 🧪 [Added] note*

- Embedding layers in PyTorch accomplish the same as linear layers that perform matrix multiplications; the reason we use embedding layers is computational efficiency
- Suppose we have 3 training examples with token IDs 2, 3, 1, and an embedding matrix with one row per possible ID:

In [13]:
# Source: Raschka, LLMs-from-scratch, embeddings-and-linear-layers.ipynb (Apache-2.0)
# Suppose we have the following 3 training examples,
# which may represent token IDs in an LLM context
idx = torch.tensor([2, 3, 1])

# The number of rows in the embedding matrix can be determined
# by obtaining the largest token ID + 1.
# If the highest token ID is 3, then we want 4 rows, for the possible
# token IDs 0, 1, 2, 3
num_idx = max(idx)+1

# The desired embedding dimension is a hyperparameter
out_dim = 5

# We use the random seed for reproducibility since
# weights in the embedding layer are initialized with
# small random values
torch.manual_seed(123)

embedding = torch.nn.Embedding(num_idx, out_dim)

print(embedding.weight)
print(embedding(torch.tensor([1])))

idx = torch.tensor([2, 3, 1])
print(embedding(idx))

Parameter containing:
tensor([[ 0.3374, -0.1778, -0.3035, -0.5880,  1.5810],
        [ 1.3010,  1.2753, -0.2010, -0.1606, -0.4015],
        [ 0.6957, -1.8061, -1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096, -0.4076,  0.7953]], requires_grad=True)
tensor([[ 1.3010,  1.2753, -0.2010, -0.1606, -0.4015]],
       grad_fn=<EmbeddingBackward0>)
tensor([[ 0.6957, -1.8061, -1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096, -0.4076,  0.7953],
        [ 1.3010,  1.2753, -0.2010, -0.1606, -0.4015]],
       grad_fn=<EmbeddingBackward0>)


Under the hood, an embedding layer is a lookup: each ID selects one row of the weight matrix.

<img src="https://github.com/chungenyu6/ai-learning-notebooks/blob/main/02-LLM/live/01-tokenization_clm_mlm/img/embedding-vs-linear_3.png?raw=true" width="450px">

*Figure: embedding lookup for IDs [2, 3, 1]. Source: Raschka, LLMs-from-scratch, embeddings-and-linear-layers bonus.*

The same result comes from an `nn.Linear` layer applied to one-hot vectors:

In [14]:
# Source: Raschka, LLMs-from-scratch, embeddings-and-linear-layers.ipynb (Apache-2.0)
onehot = torch.nn.functional.one_hot(idx)
print(onehot)

torch.manual_seed(123)
linear = torch.nn.Linear(num_idx, out_dim, bias=False)

# same small random weights as the embedding layer, so the two can be compared
linear.weight = torch.nn.Parameter(embedding.weight.T)
print(linear(onehot.float()))
print(embedding(idx))

# 🧪 [Added] assert instead of eyeballing
assert torch.allclose(linear(onehot.float()), embedding(idx))
print('one-hot @ W == embedding lookup ✓')

tensor([[0, 0, 1, 0],
        [0, 0, 0, 1],
        [0, 1, 0, 0]])
tensor([[ 0.6957, -1.8061, -1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096, -0.4076,  0.7953],
        [ 1.3010,  1.2753, -0.2010, -0.1606, -0.4015]], grad_fn=<MmBackward0>)
tensor([[ 0.6957, -1.8061, -1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096, -0.4076,  0.7953],
        [ 1.3010,  1.2753, -0.2010, -0.1606, -0.4015]],
       grad_fn=<EmbeddingBackward0>)
one-hot @ W == embedding lookup ✓


<img src="https://github.com/chungenyu6/ai-learning-notebooks/blob/main/02-LLM/live/01-tokenization_clm_mlm/img/embedding-vs-linear_4.png?raw=true" width="500px">

*Figure: one-hot × weight matrix selects the same row. Source: Raschka, LLMs-from-scratch, embeddings-and-linear-layers bonus.*

- Since all but one entry of each one-hot row is 0, the matrix multiplication is equivalent to a lookup — but wasteful, because it multiplies by zeros. `nn.Embedding` skips that work.

🧪 In a GPT, the token embedding is **added** to a positional embedding to form the input to the first block (Transformers have no built-in notion of order — see companion `04_positional_encoding_rope`):

<img src="https://github.com/chungenyu6/ai-learning-notebooks/blob/main/02-LLM/live/01-tokenization_clm_mlm/img/ch02_19.webp?raw=true" width="380px">

*Figure: the input-embedding pipeline — text → tokens → ids → token embeddings + positional embeddings. Source: Raschka, LLMs-from-scratch Ch2 (original image `ch02_compressed/19.webp`).*

## 7. What do we predict? Causal LM vs masked LM

📗 *[LLMs-from-scratch Ch2] figures; 🧪 [Added] text and code*

A GPT is trained to predict the **next** token. Each training example is a window of text; the target at every position is the next word, and the model may only use what came before it:

<img src="https://github.com/chungenyu6/ai-learning-notebooks/blob/main/02-LLM/live/01-tokenization_clm_mlm/img/ch02_12.webp?raw=true" width="550px">

*Figure: next-word prediction — the model cannot see past the target. Source: Raschka, LLMs-from-scratch Ch2 (original image `ch02_compressed/12.webp`).*

In practice, inputs and targets are the same window shifted by one token, cut from the text with a sliding window:

<img src="https://github.com/chungenyu6/ai-learning-notebooks/blob/main/02-LLM/live/01-tokenization_clm_mlm/img/ch02_13.webp?raw=true" width="550px">

*Figure: input tensor `x` and target tensor `y` = `x` shifted by one. Source: Raschka, LLMs-from-scratch Ch2 (original image `ch02_compressed/13.webp`).*

The same token ids support a different objective. BERT-style **masked language modeling** hides some tokens and predicts them from both sides:

| | **Causal LM (CLM)** — GPT, Llama | **Masked LM (MLM)** — BERT |
|---|---|---|
| Target | next token at every position | ~15% of tokens replaced by `[MASK]`; predict the originals |
| Context seen | left only (causal mask) | both sides (bidirectional) |
| Loss positions | all $T$ positions | only the masked ~15% |
| Can generate text? | yes, natively | not directly |
| Typical use | generation, chat, general-purpose LLMs | embeddings, classification, retrieval encoders |

Encoder–decoder models (T5) use a third variant: span corruption — mask spans in the input, generate them in the output.

In [15]:
# 🧪 [Added]
ids = torch.tensor(gpt2_tokenizer.encode("the cat sat on the mat because it was tired"))

# Causal LM: inputs and targets are the same sequence shifted by one.
clm_inputs, clm_targets = ids[:-1], ids[1:]
print("CLM pairs (input -> target):")
for a, b in list(zip(clm_inputs.tolist(), clm_targets.tolist()))[:5]:
    print(f"  {gpt2_tokenizer.decode([a])!r:8} -> {gpt2_tokenizer.decode([b])!r}")

# Masked LM: hide ~15% of positions; loss only where hidden (label -100 = ignore).
torch.manual_seed(0)
MASK_ID = 0                                   # stand-in id for a [MASK] token in this demo
n_mask = max(1, round(0.15 * len(ids)))
is_masked = torch.zeros(len(ids), dtype=torch.bool)
is_masked[torch.randperm(len(ids))[:n_mask]] = True
mlm_inputs = ids.masked_fill(is_masked, MASK_ID)
mlm_labels = ids.masked_fill(~is_masked, -100)
print("\nMLM input :", ["[MASK]" if m else gpt2_tokenizer.decode([i]) for i, m in zip(ids.tolist(), is_masked.tolist())])
print("MLM labels:", mlm_labels.tolist())

fake_logits = torch.zeros(len(ids), gpt2_tokenizer.n_vocab)   # an untrained model ≈ uniform prediction
print("\nln V =", round(math.log(gpt2_tokenizer.n_vocab), 3))
print("CLM loss over", len(clm_targets), "positions:", round(F.cross_entropy(fake_logits[:-1], clm_targets).item(), 3))
print("MLM loss over", int(is_masked.sum()), "positions:",
      round(F.cross_entropy(fake_logits, mlm_labels, ignore_index=-100).item(), 3))

CLM pairs (input -> target):
  'the'    -> ' cat'


  ' cat'   -> ' sat'
  ' sat'   -> ' on'
  ' on'    -> ' the'
  ' the'   -> ' mat'



MLM input : ['the', '[MASK]', ' sat', ' on', '[MASK]', ' mat', ' because', ' it', ' was', ' tired']
MLM labels: [-100, 3797, -100, -100, 262, -100, -100, -100, -100, -100]



ln V = 10.825


CLM loss over 9 positions: 10.825
MLM loss over 2 positions: 10.825


Both losses equal $\ln V \approx 10.825$ for uniform logits — a useful sanity check when training from scratch: **initial loss ≈ ln(vocab size)**.

BERT detail worth knowing: of the 15% selected positions, 80% become `[MASK]`, 10% a random token, 10% stay unchanged — so the model cannot rely on seeing `[MASK]`, which never appears at fine-tuning time.

## 8. Check yourself (interview-style)

🧪 *[Added]*

<details><summary><b>Q1. Why subword tokenization instead of words or characters?</b></summary>

Words: unbounded vocabulary, OOV tokens, no sharing of morphology. Characters/bytes: no OOV but sequences become several times longer, and attention is $O(T^2)$. Subwords give a bounded vocabulary, no OOV (byte-level BPE), and short sequences for frequent text. Vocabulary size is a knob: larger $V$ → shorter sequences but bigger embedding/output matrices and rarer, less-trained tokens.
</details>

<details><summary><b>Q2. Walk me through BPE training and encoding.</b></summary>

Train: start from the 256 byte values, repeatedly count adjacent pairs and merge the most frequent into a new token, storing the ordered merge list until the target vocab size. Encode: split text into bytes (GPT-2 first pre-splits with a regex so merges do not cross word/space/punctuation boundaries), then apply merges in learned priority order. Decode: map ids back to byte strings and concatenate. WordPiece (BERT) merges by likelihood gain instead of raw frequency; Unigram (SentencePiece) starts large and prunes.
</details>

<details><summary><b>Q3. Name some failure modes caused by tokenization.</b></summary>

Arithmetic and digit manipulation (irregular number chunks); spelling / counting letters (the model sees `strawberry` as 1–3 tokens, not 10 characters); higher cost and shorter context for non-English languages; leading-space sensitivity; glitch tokens that were in the tokenizer corpus but almost never in training data, so their embeddings stay untrained.
</details>

<details><summary><b>Q4. Why use nn.Embedding instead of one-hot × nn.Linear?</b></summary>

They compute the same thing — a one-hot row times the weight matrix selects one row — but the one-hot version materializes a $V$-dimensional vector per token and multiplies by zeros. The embedding is an indexed lookup: same result, a fraction of the memory and compute. Gradients flow only into the selected rows.
</details>

<details><summary><b>Q5. CLM vs MLM — when would you use each?</b></summary>

CLM predicts the next token with a causal mask: every position is a training signal and it can generate, so it is the objective of GPT/Llama-style LLMs. MLM predicts masked tokens with bidirectional context: better token representations for understanding tasks (classification, retrieval embeddings) but cannot generate directly and only trains on ~15% of positions.
</details>

<details><summary><b>Q6. What should the initial loss of a freshly initialized LM be?</b></summary>

About $\ln V$ (uniform prediction over the vocabulary): ≈ 10.8 for GPT-2's 50,257 tokens. A much higher value usually means the output layer is initialized too large (over-confident wrong predictions).
</details>

## References

- Raschka (2024), *Build a Large Language Model (From Scratch)*, Ch. 2; [BPE-from-scratch](https://github.com/rasbt/LLMs-from-scratch/tree/main/ch02/05_bpe-from-scratch) and [embedding-vs-matmul](https://github.com/rasbt/LLMs-from-scratch/tree/main/ch02/03_bonus_embedding-vs-matmul) bonus notebooks (Apache-2.0).
- Gage (1994), *A New Algorithm for Data Compression*; Sennrich, Haddow & Birch (2016), *Neural Machine Translation of Rare Words with Subword Units*, [arXiv:1508.07909](https://arxiv.org/abs/1508.07909).
- Radford et al. (2019), GPT-2 (byte-level BPE); Devlin et al. (2019), *BERT*, [arXiv:1810.04805](https://arxiv.org/abs/1810.04805).
- [OpenAI `tiktoken`](https://github.com/openai/tiktoken); Karpathy's [minbpe](https://github.com/karpathy/minbpe).